# EmpowerLens — D19 challenge-set scoring (Kaggle GPU)

Scores the **D12 winner** (MentalRoBERTa, binary) on the 100 justified-concern pilot texts
(`data/challenge_set/items.csv`, built by `python -m src.challenge_set sheet`), for seeds **42, 1337, 2024**:

1. train binary MentalRoBERTa with the D12 recipe (lr 3e-5, 5 epochs, weight decay 0.1, max length 512);
   D12 failed-run rule (val macro-F1 below 0.5 → re-run once);
2. fit the **D13 threshold** on val (`src.fit_binary_threshold --target-fpr 0.15`);
3. `python -m src.challenge_set predict` → `predictions_<seed>.csv` (P(distorted) + decision at that threshold);
4. delete the checkpoint (weights are never kept or committed).

The challenge set is evaluation only: nothing here trains on it. Test is not read here.
About 25–30 min per seed on a T4.

**Before running:** Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`**
(`mental/mental-roberta-base` is gated).

**Resumable:** a seed is skipped if its `predictions_<seed>.csv` is already in `/kaggle/working/d19/`.
**After it finishes:** download `d19.zip`, copy the three `predictions_<seed>.csv` into `data/challenge_set/`,
then run `venv\Scripts\python.exe -m src.challenge_set score` locally to rebuild `results.md`.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/challenge_set.py src/fit_binary_threshold.py data/challenge_set/items.csv && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = [42, 1337, 2024]
OUT    = Path("/kaggle/working/d19")
CK     = Path("/kaggle/temp/ckpt_d19")
OUT.mkdir(parents=True, exist_ok=True)

def recipe(seed):
    return (f"--model {MODEL} --seed {seed} --lr 3e-5 --epochs 5 --weight-decay 0.1 "
            f"--max-length 512 --batch-size 16 --splits data/splits")

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta(ck):
    return json.loads((Path(ck) / "meta.json").read_text())

print("config ok:", MODEL, SEEDS, OUT)

In [ ]:
# 3. Smoke test (~3 min): tiny binary run, then predict the 100 items with it. Writes nothing to OUT.
sh(f"python -m src.train_transformer --task binary --model {MODEL} --seed 42 --smoke --max-length 128 "
   f"--out /kaggle/temp/smoke")
smoke_ck = next(Path("/kaggle/temp/smoke").glob("binary_*_smoke"))   # smoke runs save as <name>_smoke
sh(f"python -m src.fit_binary_threshold --checkpoint {smoke_ck} --target-fpr 0.15")
sh(f"python -m src.challenge_set predict --checkpoint {smoke_ck} --out /kaggle/temp/smoke_pred.csv")
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True)

In [ ]:
# 4. Per seed: train binary (D12 recipe + failed-run rule), D13 threshold on val, predict the 100 items.
log = {}
for seed in SEEDS:
    pred_path = OUT / f"predictions_{seed}.csv"
    if pred_path.exists():
        print(f"seed {seed}: already done, skipping"); continue
    ck = CK / f"binary_{TAG}_{seed}"
    shutil.rmtree(CK, ignore_errors=True)
    sh(f"python -m src.train_transformer --task binary {recipe(seed)} --out {CK}")
    f1, reruns = meta(ck)["val_metrics"]["eval_macro_f1"], 0
    if f1 < 0.5:   # D12 failed-run rule: re-run once with identical settings
        print(f"[failed-run rule] seed {seed} val macro-F1 {f1:.3f} < 0.5 -> re-running once")
        shutil.rmtree(CK, ignore_errors=True)
        sh(f"python -m src.train_transformer --task binary {recipe(seed)} --out {CK}")
        f1, reruns = meta(ck)["val_metrics"]["eval_macro_f1"], 1
    sh(f"python -m src.fit_binary_threshold --checkpoint {ck} --target-fpr 0.15")
    m = meta(ck)
    sh(f"python -m src.challenge_set predict --checkpoint {ck} --out {pred_path}")
    log[seed] = {"val_macro_f1": f1, "failed_run_reruns": reruns, "binary_threshold": m["binary_threshold"],
                 "threshold_criterion": m.get("binary_threshold_criterion")}
    (OUT / f"run_{seed}.json").write_text(json.dumps(log[seed], indent=2))
    shutil.rmtree(CK, ignore_errors=True)   # weights are never kept
print(json.dumps(log, indent=2))

In [ ]:
# 5. Score (annotator part stays 'pending' here; it is filled locally) + zip for download.
for seed in SEEDS:
    shutil.copy(OUT / f"predictions_{seed}.csv", f"data/challenge_set/predictions_{seed}.csv")
sh("python -m src.challenge_set score")
shutil.copy("data/challenge_set/results.md", OUT / "results.md")
!cd /kaggle/working && rm -f d19.zip && zip -qr d19.zip d19
!ls -la /kaggle/working/d19